In [1]:
# =============================================================================
# CELL C1: Upload the 3 Downloaded Model CSVs
# =============================================================================
!pip install -q pandas nltk

from google.colab import files
import pandas as pd
import numpy as np

print("📤 Upload all 3 files now: captions_LLaVA-1.5-7B.csv, "
      "captions_Qwen2.5-VL-3B.csv, captions_Qwen3-VL-4B.csv")
uploaded = files.upload()

model_dfs = []
for filename in uploaded.keys():
    df = pd.read_csv(filename)
    model_dfs.append(df)
    print(f"✅ Loaded {filename}: {len(df)} rows | model = {df['model'].iloc[0]}")

📤 Upload all 3 files now: captions_LLaVA-1.5-7B.csv, captions_Qwen2.5-VL-3B.csv, captions_Qwen3-VL-4B.csv


Saving captions_LLaVA-1.5-7B.csv to captions_LLaVA-1.5-7B (1).csv
Saving captions_Qwen2.5-VL-3B.csv to captions_Qwen2.5-VL-3B.csv
Saving captions_Qwen3-VL-4B.csv to captions_Qwen3-VL-4B.csv
✅ Loaded captions_LLaVA-1.5-7B (1).csv: 500 rows | model = LLaVA-1.5-7B
✅ Loaded captions_Qwen2.5-VL-3B.csv: 500 rows | model = Qwen2.5-VL-3B
✅ Loaded captions_Qwen3-VL-4B.csv: 500 rows | model = Qwen3-VL-4B


In [2]:
# =============================================================================
# CELL C2: Verify All 3 Models Used the SAME 500 Real Images
# =============================================================================
image_id_sets = {df["model"].iloc[0]: set(df["image_id"].astype(str)) for df in model_dfs}
ref_set = next(iter(image_id_sets.values()))
all_match = all(s == ref_set for s in image_id_sets.values())

print(f"🔍 Same-500-images check: {'✅ PASSED' if all_match else '❌ FAILED'}")
for name, ids in image_id_sets.items():
    print(f"   {name}: {len(ids)} unique images")

assert all_match, "Models used different images — re-check RANDOM_SEED/Cell 2 consistency across notebooks."

🔍 Same-500-images check: ✅ PASSED
   LLaVA-1.5-7B: 500 unique images
   Qwen2.5-VL-3B: 500 unique images
   Qwen3-VL-4B: 500 unique images


In [3]:
# =============================================================================
# CELL C3: Combine All 3 Models Into One Dataset
# =============================================================================
combined_df = pd.concat(model_dfs, ignore_index=True)
combined_df.to_csv("combined_evaluation_dataset.csv", index=False)
print(f"✅ Combined: {len(combined_df)} rows across {combined_df['model'].nunique()} models")
files.download("combined_evaluation_dataset.csv")

✅ Combined: 1500 rows across 3 models


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# =============================================================================
# CELL C4: Standard Captioning Evaluation Metrics (model-wise, real data)
# =============================================================================
!pip install -q bert_score rouge_score pycocoevalcap nltk

import re, warnings, time
warnings.filterwarnings("ignore")
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score
from pycocoevalcap.cider.cider import Cider
from tqdm.auto import tqdm

bleu_smoothing = SmoothingFunction().method4
rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
cider_scorer = Cider()

def compute_bleu(candidate, references):
    cand_tokens = word_tokenize(candidate.lower())
    ref_tokens_list = [word_tokenize(r.lower()) for r in references]
    return sentence_bleu(ref_tokens_list, cand_tokens, smoothing_function=bleu_smoothing)

def compute_rouge(candidate, references):
    best_r1, best_rl = 0.0, 0.0
    for ref in references:
        s = rouge.score(ref, candidate)
        best_r1 = max(best_r1, s["rouge1"].fmeasure)
        best_rl = max(best_rl, s["rougeL"].fmeasure)
    return best_r1, best_rl

def clean_tokenize_sentence(text):
    text = re.sub(r"[^a-zA-Z0-9\s]", "", str(text).lower())
    return " ".join(text.split())

def build_cider_inputs(df, caption_col):
    gts, res = {}, {}
    for _, row in df.iterrows():
        img_key = f"{row['model']}_{row['image_id']}"
        refs = row["reference_captions"].split(" | ")
        gts[img_key] = [clean_tokenize_sentence(r) for r in refs]
        res[img_key] = [clean_tokenize_sentence(row[caption_col])]
    return gts, res

metric_rows = []
model_names = list(combined_df["model"].unique())
job_bar = tqdm(total=len(model_names) * 2, desc="📊 Overall metrics progress", unit="job")
t0 = time.time()

for model_name in model_names:
    model_df = combined_df[combined_df["model"] == model_name].reset_index(drop=True)

    for caption_type, caption_col in [("Baseline", "baseline_caption"), ("Grounded", "grounded_caption")]:
        candidates = model_df[caption_col].astype(str).tolist()
        references_list = [row.split(" | ") for row in model_df["reference_captions"].astype(str).tolist()]

        bleu_scores = [
            compute_bleu(c, r) for c, r in
            tqdm(zip(candidates, references_list), total=len(candidates),
                 desc=f"  BLEU [{model_name}/{caption_type}]", leave=False)
        ]
        avg_bleu = float(np.mean(bleu_scores))

        r1s, rls = [], []
        for c, r in tqdm(zip(candidates, references_list), total=len(candidates),
                          desc=f"  ROUGE [{model_name}/{caption_type}]", leave=False):
            r1, rl = compute_rouge(c, r)
            r1s.append(r1); rls.append(rl)
        avg_rouge1, avg_rougeL = float(np.mean(r1s)), float(np.mean(rls))

        P, R, F1 = bertscore_score(candidates, references_list, lang="en", verbose=True, rescale_with_baseline=True)
        avg_bertscore_f1 = float(F1.mean().item())

        gts, res = build_cider_inputs(model_df, caption_col)
        cider_score, _ = cider_scorer.compute_score(gts, res)

        metric_rows.append({
            "Model": model_name,
            "Caption Type": caption_type,
            "N Images": len(candidates),
            "BLEU-4": round(avg_bleu, 4),
            "ROUGE-1": round(avg_rouge1, 4),
            "ROUGE-L": round(avg_rougeL, 4),
            "BERTScore-F1": round(avg_bertscore_f1, 4),
            "CIDEr": round(float(cider_score), 4),
        })
        job_bar.update(1)

job_bar.close()
print(f"✅ All metrics computed in {(time.time()-t0)/60:.1f} minutes")

standard_metrics_df = pd.DataFrame(metric_rows)
standard_metrics_df.to_csv("standard_captioning_metrics_table.csv", index=False)
files.download("standard_captioning_metrics_table.csv")
print(standard_metrics_df.to_string(index=False))

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 9.2 MB/s eta 0:00:00


📊 Overall metrics progress:   0%|          | 0/6 [00:00<?, ?job/s]

  BLEU [LLaVA-1.5-7B/Baseline]:   0%|          | 0/500 [00:00<?, ?it/s]

  ROUGE [LLaVA-1.5-7B/Baseline]:   0%|          | 0/500 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

  0%|          | 0/40 [00:00<?, ?it/s]

done in 15.28 seconds, 163.70 sentences/sec
✅ All metrics computed in 2.4 minutes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

        Model Caption Type  N Images  BLEU-4  ROUGE-1  ROUGE-L  BERTScore-F1  CIDEr
 LLaVA-1.5-7B     Baseline       500  0.0647   0.2547   0.2186        0.3499 0.0003
 LLaVA-1.5-7B     Grounded       500  0.0711   0.2741   0.2367        0.3697 0.0015
Qwen2.5-VL-3B     Baseline       500  0.1883   0.4955   0.4416        0.5542 0.5833
Qwen2.5-VL-3B     Grounded       500  0.0575   0.2613   0.2173        0.3208 0.0008
  Qwen3-VL-4B     Baseline       500  0.0712   0.2706   0.2300        0.3551 0.0238
  Qwen3-VL-4B     Grounded       500  0.0500   0.2254   0.1903        0.2878 0.0001


In [5]:
# =============================================================================
# CELL C5: Final Research-Format Table (paper-ready) + Download
# =============================================================================
pivot_df = standard_metrics_df.pivot(
    index="Model", columns="Caption Type",
    values=["BLEU-4", "ROUGE-1", "ROUGE-L", "BERTScore-F1", "CIDEr"]
)
pivot_df.columns = [f"{metric} ({ctype})" for metric, ctype in pivot_df.columns]
pivot_df = pivot_df.reset_index()

print("\n" + "=" * 100)
print("📊 FINAL RESEARCH-FORMAT TABLE — Model-wise Evaluation (500 real COCO 2014 images)")
print("=" * 100)
print(pivot_df.to_string(index=False))

pivot_df.to_csv("final_research_table.csv", index=False)
with open("final_research_table.md", "w") as f:
    f.write(pivot_df.to_markdown(index=False))
with open("final_research_table.tex", "w") as f:
    f.write(pivot_df.to_latex(index=False, float_format="%.4f"))

for fname in ["final_research_table.csv", "final_research_table.md", "final_research_table.tex"]:
    files.download(fname)

print("\n🎉 Project complete — real 500 COCO images, 3 real models, 0 dummy data.")


📊 FINAL RESEARCH-FORMAT TABLE — Model-wise Evaluation (500 real COCO 2014 images)
        Model  BLEU-4 (Baseline)  BLEU-4 (Grounded)  ROUGE-1 (Baseline)  ROUGE-1 (Grounded)  ROUGE-L (Baseline)  ROUGE-L (Grounded)  BERTScore-F1 (Baseline)  BERTScore-F1 (Grounded)  CIDEr (Baseline)  CIDEr (Grounded)
 LLaVA-1.5-7B             0.0647             0.0711              0.2547              0.2741              0.2186              0.2367                   0.3499                   0.3697            0.0003            0.0015
Qwen2.5-VL-3B             0.1883             0.0575              0.4955              0.2613              0.4416              0.2173                   0.5542                   0.3208            0.5833            0.0008
  Qwen3-VL-4B             0.0712             0.0500              0.2706              0.2254              0.2300              0.1903                   0.3551                   0.2878            0.0238            0.0001


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Project complete — real 500 COCO images, 3 real models, 0 dummy data.
